# 写出 LAMMPS C4 表势 `*.table`

12-6-4 中的 $-C_4/r^4$ 项用 `pair_style hybrid/overlay … table …` 叠加。本 Notebook 按给定 **C4**、**cutoff** 生成 LAMMPS `pair_style table` 文件：

$$
E(r)=-\frac{C_4}{r^4},\qquad F(r)=-\frac{\mathrm{d}E}{\mathrm{d}r}=-\frac{4C_4}{r^5}
$$

单位与 `units real` / Amber 注释一致：**$C_4$ 为 kcal·mol⁻¹·Å⁴**，距离 Å。`pair_coeff` 中的 section 名默认 `C4_POT`。

In [ ]:
# ---- 参数：改这里即可 ----
name = "C4_Cl_Ow"       # 输出 {name}.table；勿带 .table 后缀
C4 = -55             # kcal/mol/Å^4（.ff 行尾注释） 
#112.2 for Mg；-55 for Cl
cutoff = 12.0           # Å；与 lj/cut/coul/long 截断对齐
r_min = 0.5             # Å；表内最小 r（须小于可能出现的最近邻）
n_points = 1000         # 表点数；建议与 pair_style table … N 一致
keyword = "C4_POT"      # pair_coeff 里的 section 名
out_dir = "."           # 输出目录

In [ ]:
from pathlib import Path

import numpy as np


def write_c4_table(
    name,
    C4,
    cutoff,
    *,
    r_min=0.5,
    n_points=1000,
    keyword="C4_POT",
    out_dir=".",
):
    """Write LAMMPS pair_style table for E = -C4/r^4 (units real).

    Parameters
    ----------
    name : str
        Base filename; writes ``{name}.table`` (``.table`` stripped if present).
    C4 : float
        Coefficient in kcal/mol/Å^4.
    cutoff : float
        Outer distance of the table (Å).
    r_min, n_points, keyword, out_dir
        Table grid, section keyword for ``pair_coeff``, and output folder.

    Returns
    -------
    pathlib.Path
        Path of the written table file.
    """
    stem = Path(str(name)).name
    if stem.lower().endswith(".table"):
        stem = stem[: -len(".table")]
    if not stem:
        raise ValueError("name must be a non-empty filename stem")

    C4 = float(C4)
    cutoff = float(cutoff)
    r_min = float(r_min)
    n_points = int(n_points)
    if C4 == 0.0:
        raise ValueError("C4 is 0; table would be identically zero")
    if cutoff <= r_min:
        raise ValueError(f"cutoff ({cutoff}) must be > r_min ({r_min})")
    if n_points < 2:
        raise ValueError("n_points must be >= 2")
    if r_min <= 0.0:
        raise ValueError("r_min must be > 0")

    r = np.linspace(r_min, cutoff, n_points)
    energy = -C4 / r**4
    force = -4.0 * C4 / r**5  # F = -dE/dr

    out_path = Path(out_dir).expanduser().resolve() / f"{stem}.table"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    lines = [
        f"# DATE: generated by simul_model.ipynb",
        f"# UNITS: real",
        f"# E = -C4/r^4 ; F = -dE/dr = -4*C4/r^5",
        f"# C4 = {C4} kcal/mol/Angstrom^4",
        f"# r_min = {r_min} Angstrom ; cutoff = {cutoff} Angstrom ; N = {n_points}",
        "",
        str(keyword),
        f"N {n_points} R {r_min:.10g} {cutoff:.10g}",
        "",
    ]
    for i, (ri, ei, fi) in enumerate(zip(r, energy, force), start=1):
        lines.append(f"{i} {ri:.10g} {ei:.10g} {fi:.10g}")
    lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"wrote {out_path}")
    print(
        f"  C4={C4}  r=[{r_min}, {cutoff}]  N={n_points}  "
        f"keyword={keyword!r}"
    )
    print(
        "  pair_coeff example:\n"
        f"    pair_coeff  I  J  table  {out_path.name}  {keyword}"
    )
    return out_path


write_c4_table(
    name,
    C4,
    cutoff,
    r_min=r_min,
    n_points=n_points,
    keyword=keyword,
    out_dir=out_dir,
)